# GNSM — Grounded Narrative State Model (Colab runner)

Clone → bootstrap → run. Set **Runtime → Change runtime type → GPU** (T4 is enough) before you start, then run the cells top to bottom.

1. GPU check
2. Clone the repo
3. One-shot bootstrap (installs the CUDA stack, prints a runtime report)
4. No-download reference demo
5. GPU neural smoke (trains the state stack on a synthetic batch)
6. Stage-0 decodability probe
7. *(optional)* a real frozen Hugging Face generator

## 1 · Confirm a GPU is attached

In [ ]:
!nvidia-smi

## 2 · Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/GIND123/Dynamic-Narration-Graph.git"
REPO_DIR = "Dynamic-Narration-Graph"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL
%cd $REPO_DIR

## 3 · Bootstrap the environment

Keeps Colab's CUDA-matched torch, installs the rest of the stack, and prints a `gnsm doctor` report. Takes ~1–2 minutes the first time.

In [ ]:
!python -m gnsm.colab.bootstrap

## 4 · No-download reference demo

Runs the full extract → encode → transition → generate → verify loop with deterministic components (no model weights).

In [ ]:
!python -m gnsm demo

## 5 · GPU neural smoke

Builds the real `GraphStateEncoder` + grounded decode heads + `NeuralTransitionModel`, and trains them on a fixed synthetic batch. If `device=cuda` and the loss drops, the GPU training path works end-to-end.

In [ ]:
!python -m gnsm smoke --json --steps 80

## 6 · Stage-0 decodability probe (go / no-go)

The cheapest experiment in the plan: is narrative state linearly decodable to labels? We synthesize separable features here; swap in cached `z_t` features and real labels for the actual P0 study.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n, d, k = 240, 32, 3
labels = rng.integers(0, k, size=n)
centers = rng.normal(scale=3.0, size=(k, d))
features = centers[labels] + rng.normal(scale=1.0, size=(n, d))
np.save("features.npy", features.astype("float32"))
np.save("labels.npy", labels.astype(str))

!python -m gnsm.training.stage0_probe features.npy labels.npy

## 7 · (Optional) A real frozen Hugging Face generator

Wires a frozen HF model into the GNSM generation plane. The default is a tiny ungated model so it just runs. For the headline generators (Llama-3.1-8B / Qwen2.5-14B) set `quantize=True` and provide an `HF_TOKEN` in Colab **Secrets** (the bootstrap loads it automatically).

In [ ]:
from gnsm.generation.huggingface import HuggingFaceFrozenGenerator
from gnsm.pipeline import GNSMSystem
from gnsm.schemas import PlotAction

# Tiny + ungated so this cell runs with no token. For a headline model swap in:
#   HuggingFaceFrozenGenerator("meta-llama/Llama-3.1-8B-Instruct", quantize=True)
generator = HuggingFaceFrozenGenerator(
    "Qwen/Qwen2.5-0.5B-Instruct", max_new_tokens=200, quantize=False
)

system = GNSMSystem.reference(generator=generator)
scene = 'Mara is alive. Mara waited in the observatory. Mara said, "The signal is back."'
state = system.initialize(scene)
participants = tuple(eid for eid, e in state.graph.entities.items() if e.name == "Mara")
result = system.generate_next(
    previous_scene=scene,
    state=state,
    action=PlotAction(
        intent="Mara studies the returned signal without leaving.", participants=participants
    ),
    next_scene_id="scene-1",
    rolling_summary="Mara detected a signal in the observatory.",
)
print("ACCEPTED:", result.verification.accepted, "| ATTEMPTS:", result.attempts)
print("-" * 60)
print(result.text)